# Phase 2 — ResNet-34 + U-Net Colorization

**Task:** Same as Phase 1 — predict ab channels from the L channel.  
**Architecture:** ResNet-34 encoder (ImageNet pretrained, frozen) + U-Net decoder.  
**Loss:** L1 on ab channels + VGG-16 perceptual loss (relu2_2 + relu3_3).  
**Color space:** LAB — encoder sees L repeated 3×; decoder predicts `a` and `b`.

---

## Project Overview — 3-Phase Colorization Pipeline

This project trains a deep learning model to colorize grayscale images in three progressively more powerful phases. Each phase produces increasingly realistic colors.

| Phase | Notebook | Architecture | Loss | Dependency |
|---|---|---|---|---|
| **1** | `01_unet.ipynb` | U-Net from scratch | L1 | None — fully independent baseline |
| **2 (this)** | `02_resnet_unet.ipynb` | ResNet-34 encoder + U-Net decoder | L1 + Perceptual | None — independent (uses ImageNet weights) |
| **3** | `03_cgan.ipynb` | Phase 2 generator + PatchGAN discriminator | L1 + Perceptual + Adversarial | Requires Phase 2 `best.pth` |

**Phase 1 — Baseline:** Trains a vanilla U-Net entirely from scratch. No pretrained weights, pure L1 pixel loss. The model will correctly learn *where* colors go (sky is blue, grass is green) but L1 loss minimizes average error, which causes the network to predict the safe mean color rather than committing to a vivid one. Expect correct spatial structure but washed-out, desaturated outputs. This phase exists as the reference point to measure how much each subsequent phase improves things.

**Phase 2 — Transfer Learning (this notebook):** Swaps the encoder for a ResNet-34 pretrained on ImageNet. The frozen encoder supplies rich semantic features (textures, object boundaries) from epoch 1, so the decoder starts from a much stronger base. Adding a perceptual loss (VGG feature matching) pushes the model to match high-level structure, not just pixel values. Colors become more saturated and better localized, and convergence is 3–5× faster.

**Phase 3 — cGAN:** Adds a PatchGAN discriminator that evaluates whether each 70×70 patch of the colorized image looks real. The adversarial loss eliminates the muddy, averaged colors that L1 alone produces — the generator is now penalized for *any* locally unconvincing output, pushing it toward vivid, photorealistic colorization. Must be initialized from Phase 2 weights; starting from scratch makes GAN training unstable.

## 1. Setup

In [15]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [21]:
import ssl                                                                 
import urllib.request                                                      
                                                                             
ssl._create_default_https_context = ssl._create_unverified_context         
opener = urllib.request.build_opener(                                      
      urllib.request.HTTPSHandler(context=ssl._create_unverified_context())  
  )
urllib.request.install_opener(opener)

In [22]:
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

from src.data.dataset import ColorizationDataset, lab_to_rgb
from src.models.resnet_unet import ResNetUNet
from src.losses.perceptual import PerceptualLoss


print(f"PyTorch {torch.__version__}")
DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Device: {DEVICE}")

PyTorch 2.11.0
Device: mps


## 2. Configuration

In [23]:
CFG = {
    # Data
    "processed_dir": PROJECT_ROOT / "data" / "processed",
    "batch_size": 32,
    "num_workers": 4,
    "train_subset": 15000,       # set to None to train on the full split
    "subset_seed": 42,        # reproducible subset selection

    # Training
    "epochs": 20,
    "lr": 2e-4,
    "betas": (0.5, 0.999),
    "lr_decay_start": 10,

    # Loss weights
    "lambda_l1": 1.0,
    "lambda_perceptual": 0.1,   # perceptual magnitudes are ~10x larger than L1

    # Checkpointing
    "checkpoint_dir": PROJECT_ROOT / "checkpoints" / "resnet_unet",
    "save_every": 5,
}

## 3. Dataset & DataLoader

In [24]:
import pandas as pd

manifest_path = CFG["processed_dir"] / "manifest.csv"
_manifest = pd.read_csv(manifest_path)

_train_pool = _manifest[_manifest["split"] == "train"]
if CFG["train_subset"] is not None and CFG["train_subset"] < len(_train_pool):
    train_filenames = _train_pool.sample(
        n=CFG["train_subset"], random_state=CFG["subset_seed"]
    )["filename"].tolist()
    print(f"Training on subset of {len(train_filenames):,} images "
          f"(seed={CFG['subset_seed']}, full split has {len(_train_pool):,}).")
else:
    train_filenames = None
    print(f"Training on the full split of {len(_train_pool):,} images.")

train_ds = ColorizationDataset("train", processed_dir=CFG["processed_dir"],
                                horizontal_flip=True, filenames=train_filenames)
val_ds   = ColorizationDataset("val",   processed_dir=CFG["processed_dir"], horizontal_flip=False)
test_ds  = ColorizationDataset("test",  processed_dir=CFG["processed_dir"], horizontal_flip=False)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,
                          num_workers=CFG["num_workers"], pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"], shuffle=False,
                          num_workers=CFG["num_workers"], pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG["batch_size"], shuffle=False,
                          num_workers=CFG["num_workers"], pin_memory=True)

print(f"Train: {len(train_ds):,}  |  Val: {len(val_ds):,}  |  Test: {len(test_ds):,}")


Train: 74,919  |  Val: 9,222  |  Test: 9,321


## 4. Model

In [25]:
model = ResNetUNet(out_channels=2, freeze_encoder=True).to(DEVICE)

total_params    = sum(p.numel() for p in model.parameters())
trainable       = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen          = total_params - trainable
print(f"Parameters: {total_params:,} total  |  {trainable:,} trainable  |  {frozen:,} frozen")

with torch.no_grad():
    dummy = torch.randn(2, 1, 256, 256, device=DEVICE)
    out   = model(dummy)
    print(f"Input shape:  {dummy.shape}")
    print(f"Output shape: {out.shape}   (expected: (2, 2, 256, 256))")
    print(f"Output range: [{out.min():.3f}, {out.max():.3f}]  (expected: [-1, 1])")

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /Users/iquenavarro/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:01<00:00, 70.0MB/s]


Parameters: 24,828,738 total  |  3,544,066 trainable  |  21,284,672 frozen
Input shape:  torch.Size([2, 1, 256, 256])
Output shape: torch.Size([2, 2, 256, 256])   (expected: (2, 2, 256, 256))
Output range: [-1.000, 1.000]  (expected: [-1, 1])


## 5. Loss Functions

In [26]:
l1_criterion          = nn.L1Loss()
perceptual_criterion  = PerceptualLoss().to(DEVICE)

print("L1 loss: ready")
print(f"Perceptual loss: VGG-16 relu2_2 + relu3_3, frozen ({sum(p.numel() for p in perceptual_criterion.parameters()):,} params)")

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /Users/iquenavarro/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:05<00:00, 94.3MB/s] 


L1 loss: ready
Perceptual loss: VGG-16 relu2_2 + relu3_3, frozen (1,735,488 params)


## 6. Training

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=CFG["lr"], betas=CFG["betas"])

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CFG["epochs"] - CFG["lr_decay_start"],
    eta_min=1e-6,
)

In [ ]:
def train_epoch(model, loader, l1_crit, perc_crit, optimizer, device):
    model.train()
    total_loss = total_l1 = total_perc = 0.0
    pbar = tqdm(loader, desc="  train", leave=False, unit="batch")
    for l_batch, ab_batch in pbar:
        l_batch  = l_batch.to(device)
        ab_batch = ab_batch.to(device)

        optimizer.zero_grad()
        pred_ab = model(l_batch)

        l1_loss   = l1_crit(pred_ab, ab_batch)
        perc_loss = perc_crit(l_batch, pred_ab, l_batch, ab_batch)
        loss      = CFG["lambda_l1"] * l1_loss + CFG["lambda_perceptual"] * perc_loss

        loss.backward()
        optimizer.step()

        n = l_batch.size(0)
        total_loss += loss.item() * n
        total_l1   += l1_loss.item() * n
        total_perc += perc_loss.item() * n
        pbar.set_postfix(loss=f"{loss.item():.4f}", l1=f"{l1_loss.item():.4f}")

    N = len(loader.dataset)
    return total_loss / N, total_l1 / N, total_perc / N


@torch.no_grad()
def eval_epoch(model, loader, l1_crit, perc_crit, device):
    model.eval()
    total_loss = total_l1 = total_perc = total_psnr = 0.0
    for l_batch, ab_batch in tqdm(loader, desc="  val  ", leave=False, unit="batch"):
        l_batch  = l_batch.to(device)
        ab_batch = ab_batch.to(device)

        pred_ab   = model(l_batch)
        l1_loss   = l1_crit(pred_ab, ab_batch)
        perc_loss = perc_crit(l_batch, pred_ab, l_batch, ab_batch)
        loss      = CFG["lambda_l1"] * l1_loss + CFG["lambda_perceptual"] * perc_loss

        n = l_batch.size(0)
        total_loss += loss.item() * n
        total_l1   += l1_loss.item() * n
        total_perc += perc_loss.item() * n
        total_psnr += psnr(pred_ab.cpu(), ab_batch.cpu()) * n

    N = len(loader.dataset)
    return total_loss / N, total_l1 / N, total_perc / N, total_psnr / N

In [ ]:
CFG["checkpoint_dir"].mkdir(parents=True, exist_ok=True)

steps_per_epoch = len(train_loader)
print(f"Phase 2 — ResNet-34 encoder + U-Net decoder (transfer learning)")
print(f"  {CFG['epochs']} epochs x {steps_per_epoch:,} steps/epoch = {CFG['epochs'] * steps_per_epoch:,} total steps")
print(f"  Device: {DEVICE}  |  Batch size: {CFG['batch_size']}  |  lr: {CFG['lr']}")
print(f"  Loss weights: L1={CFG['lambda_l1']}  Perceptual={CFG['lambda_perceptual']}")
print(f"  Checkpoints -> {CFG['checkpoint_dir']}\n")

history = {"train_loss": [], "train_l1": [], "train_perc": [],
           "val_loss": [],   "val_l1": [],   "val_perc": [], "val_psnr": []}
best_val_loss = float("inf")

for epoch in range(1, CFG["epochs"] + 1):
    t0 = time.time()

    train_loss, train_l1, train_perc = train_epoch(
        model, train_loader, l1_criterion, perceptual_criterion, optimizer, DEVICE
    )
    val_loss, val_l1, val_perc, val_psnr = eval_epoch(
        model, val_loader, l1_criterion, perceptual_criterion, DEVICE
    )

    if epoch >= CFG["lr_decay_start"]:
        scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_l1"].append(train_l1)
    history["train_perc"].append(train_perc)
    history["val_loss"].append(val_loss)
    history["val_l1"].append(val_l1)
    history["val_perc"].append(val_perc)
    history["val_psnr"].append(val_psnr)

    elapsed = time.time() - t0
    eta_s   = elapsed * (CFG["epochs"] - epoch)
    eta_str = f"{eta_s/3600:.1f}h" if eta_s > 3600 else f"{eta_s/60:.0f}min"
    lr_now  = optimizer.param_groups[0]["lr"]

    print(
        f"[{epoch:3d}/{CFG['epochs']}]  "
        f"loss={val_loss:.4f}  l1={val_l1:.4f}  perc={val_perc:.4f}  "
        f"PSNR={val_psnr:.2f}dB  lr={lr_now:.2e}  "
        f"{elapsed:.0f}s/epoch  ETA {eta_str}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), CFG["checkpoint_dir"] / "best.pth")
        print(f"         -> New best! val_loss={best_val_loss:.4f}  saved best.pth")

    if epoch % CFG["save_every"] == 0:
        ckpt_name = f"epoch_{epoch:03d}.pth"
        torch.save(
            {"epoch": epoch, "model": model.state_dict(),
             "optimizer": optimizer.state_dict(), "history": history},
            CFG["checkpoint_dir"] / ckpt_name,
        )
        print(f"         -> Checkpoint saved: {ckpt_name}")

print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")

## 7. Training Curves

In [ ]:
epochs_range = range(1, len(history["train_loss"]) + 1)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(17, 4))

ax1.plot(epochs_range, history["train_loss"], label="Train")
ax1.plot(epochs_range, history["val_loss"],   label="Val")
ax1.set_title("Total Loss (λ·L1 + λ·Perceptual)")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history["train_l1"],   label="Train L1")
ax2.plot(epochs_range, history["val_l1"],     label="Val L1")
ax2.plot(epochs_range, history["train_perc"], label="Train Perc", linestyle="--")
ax2.plot(epochs_range, history["val_perc"],   label="Val Perc",   linestyle="--")
ax2.set_title("L1 vs Perceptual Loss")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss")
ax2.legend(); ax2.grid(True, alpha=0.3)

ax3.plot(epochs_range, history["val_psnr"], color="tab:green", label="Val PSNR")
ax3.set_title("Validation PSNR (ab channels)")
ax3.set_xlabel("Epoch"); ax3.set_ylabel("PSNR (dB)")
ax3.legend(); ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Visualize Colorization Results

Each row: **Grayscale input → ResNet-UNet prediction → Ground truth**.

In [ ]:
model.load_state_dict(torch.load(CFG["checkpoint_dir"] / "best.pth", map_location=DEVICE))
model.eval()
print("Loaded best checkpoint.")

In [ ]:
@torch.no_grad()
def visualize_predictions(model, loader, device, n=8, title="ResNet-UNet colorization"):
    l_batch, ab_batch = next(iter(loader))
    l_batch, ab_batch = l_batch[:n].to(device), ab_batch[:n].to(device)

    pred_ab  = model(l_batch).cpu()
    l_batch  = l_batch.cpu()
    ab_batch = ab_batch.cpu()

    fig, axes = plt.subplots(3, n, figsize=(2.5 * n, 8))
    fig.suptitle(title, fontsize=13)
    row_labels = ["Grayscale (input)", "Predicted (ResNet-UNet)", "Ground truth"]

    for i in range(n):
        gray      = (l_batch[i, 0].numpy() + 1.0) / 2.0
        pred_rgb  = lab_to_rgb(l_batch[i], pred_ab[i])
        truth_rgb = lab_to_rgb(l_batch[i], ab_batch[i])

        axes[0, i].imshow(gray, cmap="gray", vmin=0, vmax=1)
        axes[1, i].imshow(pred_rgb)
        axes[2, i].imshow(truth_rgb)

        for row in range(3):
            axes[row, i].axis("off")
            if i == 0:
                axes[row, i].set_ylabel(row_labels[row], fontsize=9)

    plt.tight_layout()
    plt.show()

visualize_predictions(model, val_loader, DEVICE)

## 9. Test-set Evaluation

In [ ]:
test_loss, test_l1, test_perc, test_psnr = eval_epoch(
    model, test_loader, l1_criterion, perceptual_criterion, DEVICE
)
print(f"Test total loss : {test_loss:.4f}")
print(f"Test L1         : {test_l1:.4f}")
print(f"Test Perceptual : {test_perc:.4f}")
print(f"Test PSNR       : {test_psnr:.2f} dB")

## 10. Checkpoint Info

In [ ]:
# To resume training or fine-tune with unfrozen encoder:
#
# ckpt = torch.load(CFG["checkpoint_dir"] / "epoch_010.pth", map_location=DEVICE)
# model.load_state_dict(ckpt["model"])
# optimizer.load_state_dict(ckpt["optimizer"])
# history = ckpt["history"]
# start_epoch = ckpt["epoch"] + 1
#
# To unfreeze the ResNet encoder for fine-tuning (use a lower lr):
# model.unfreeze_encoder()
# optimizer = torch.optim.Adam(model.parameters(), lr=5e-5, betas=CFG["betas"])
print("Checkpoints saved to:", CFG["checkpoint_dir"])
print("  best.pth           — lowest val loss")
print("  epoch_XXX.pth      — full state (model + optimizer + history) every", CFG["save_every"], "epochs")